# Notebook 3 · Distribuciones de probabilidad
**Consumo de energía eléctrica – Electro Puno S.A.A.**

Objetivo: ajustar distribuciones teóricas a las variables clave para generar **nuevos indicadores**:
1. Ajuste de distribuciones al **consumo por cliente residencial (BT5B)**: Normal, Log-normal, Gamma, Weibull, Exponencial.
2. Selección del mejor modelo (AIC, BIC, prueba de Kolmogorov-Smirnov, gráfico Q-Q).
3. Indicadores probabilísticos: P(consumo ≤ 30 kWh), P(consumo ≤ 100 kWh – FOSE), percentiles.
4. Comparación entre provincias (ajuste por provincia, Kruskal-Wallis, intervalos bootstrap).
5. **Detección de anomalías** de consumo y de precio a partir de las distribuciones.
6. Ley de tamaño de los distritos (rango-tamaño / Zipf).

## 1. Carga

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

pd.set_option("display.float_format", "{:,.4f}".format)
OUT = Path("data/processed")
rng = np.random.default_rng(42)

df = pd.read_parquet(OUT / "consumo_limpio.parquet")
ind = pd.read_csv(OUT / "indicadores_distrito.csv", dtype={"ubigeo": str, "geo_id": str})

b5 = df[(df["tarifa"] == "BT5B") & (df["clientes"] > 0) & (df["consumo_kwh"] > 0)].copy()
x = b5["kwh_cliente"].to_numpy()
print(f"Registros BT5B con consumo: {len(b5):,} (agrupan {b5['clientes'].sum():,} clientes)")
pd.Series(x).describe(percentiles=[.05, .25, .5, .75, .95, .99])

Registros BT5B con consumo: 12,394 (agrupan 14,527,971 clientes)


count   12,394.0000
mean        50.1027
std        154.9107
min          0.0488
5%           2.0000
25%          9.0000
50%         17.9292
75%         35.5000
95%        149.3050
99%        839.3500
max      3,588.5000
dtype: float64

## 2. Ajuste de distribuciones al consumo por cliente (kWh/mes)
Se estiman los parámetros por **máxima verosimilitud** (`scipy.stats.<dist>.fit`). Para las distribuciones
definidas en valores positivos se fija la localización en 0.

In [2]:
candidatas = {
    "Normal": (stats.norm, {}),
    "Log-normal": (stats.lognorm, {"floc": 0}),
    "Gamma": (stats.gamma, {"floc": 0}),
    "Weibull": (stats.weibull_min, {"floc": 0}),
    "Exponencial": (stats.expon, {"floc": 0}),
}

ajustes, filas = {}, []
for nombre, (dist, kw) in candidatas.items():
    params = dist.fit(x, **kw)
    ll = np.sum(dist.logpdf(x, *params))
    k = len(params) - len(kw)
    ks = stats.kstest(x, dist.cdf, args=params)
    ajustes[nombre] = (dist, params)
    filas.append({"distribucion": nombre, "parametros": np.round(params, 4).tolist(), "logL": ll,
                  "AIC": 2 * k - 2 * ll, "BIC": k * np.log(len(x)) - 2 * ll,
                  "KS_D": ks.statistic, "KS_pvalor": ks.pvalue})
tabla = pd.DataFrame(filas).sort_values("AIC").reset_index(drop=True)
mejor = tabla.loc[0, "distribucion"]
print("Mejor ajuste según AIC:", mejor)
tabla

Mejor ajuste según AIC: Log-normal


,distribucion,parametros,logL,AIC,BIC,KS_D,KS_pvalor
0,Log-normal,"[1.273, 0.0, 17.9209]","-56,347.0106","112,698.0212","112,712.8711",0.0465,0.0000
1,Weibull,"[0.6935, 0.0, 34.5002]","-58,328.5097","116,661.0194","116,675.8694",0.1126,0.0000
2,Gamma,"[0.6009, 0.0, 83.3854]","-59,584.9111","119,173.8221","119,188.6721",0.1754,0.0000
3,Exponencial,"[0.0, 50.1027]","-60,905.0516","121,812.1032","121,819.5282",0.2551,0.0000
4,Normal,"[50.1027, 154.9044]","-80,086.8907","160,177.7814","160,192.6314",0.3738,0.0000


> Con miles de observaciones el p-valor de KS casi siempre es pequeño (la prueba detecta diferencias mínimas);
> por eso se usa el estadístico **D** y el **AIC** para *comparar* modelos, y el gráfico Q-Q para validar visualmente.

In [3]:
lim = np.quantile(x, 0.99)
grid = np.linspace(0.01, lim, 400)
fig = go.Figure()
fig.add_histogram(x=x[x <= lim], histnorm="probability density", nbinsx=80, name="Datos", opacity=0.45)
for nombre, (dist, params) in ajustes.items():
    fig.add_scatter(x=grid, y=dist.pdf(grid, *params), name=nombre,
                    line=dict(width=4 if nombre == mejor else 1.5))
fig.add_vline(x=100, line_dash="dash", line_color="red", annotation_text="100 kWh (FOSE)")
fig.update_layout(title="Consumo por cliente BT5B: histograma vs distribuciones ajustadas (hasta P99)",
                  xaxis_title="kWh por cliente al mes", yaxis_title="densidad")
fig.show()

In [4]:
dist_m, par_m = ajustes[mejor]
probs = (np.arange(1, len(x) + 1) - 0.5) / len(x)
teo = dist_m.ppf(probs, *par_m)
emp = np.sort(x)
fig = px.scatter(x=teo, y=emp, log_x=True, log_y=True, opacity=0.4,
                 labels={"x": f"Cuantiles teóricos ({mejor})", "y": "Cuantiles observados"},
                 title=f"Gráfico Q-Q – {mejor}")
fig.add_scatter(x=[emp.min() + 0.1, emp.max()], y=[emp.min() + 0.1, emp.max()], mode="lines",
                name="y = x", line=dict(color="red", dash="dash"))
fig.show()

## 3. Indicadores probabilísticos
Con el modelo ajustado se calculan probabilidades útiles para la toma de decisiones y se comparan con las empíricas.

In [5]:
def resumen_prob(dist, params, datos):
    return pd.DataFrame({
        "indicador": ["P(X ≤ 30 kWh) – bajo consumo", "P(X ≤ 100 kWh) – rango FOSE", "P(X > 300 kWh) – alto consumo",
                      "Mediana (kWh)", "Percentil 90 (kWh)", "Percentil 99 (kWh)", "Media (kWh)"],
        "modelo": [dist.cdf(30, *params), dist.cdf(100, *params), dist.sf(300, *params),
                   dist.ppf(.5, *params), dist.ppf(.9, *params), dist.ppf(.99, *params), dist.mean(*params)],
        "empirico": [np.mean(datos <= 30), np.mean(datos <= 100), np.mean(datos > 300),
                     np.median(datos), np.quantile(datos, .9), np.quantile(datos, .99), datos.mean()],
    })
ind_prob = resumen_prob(dist_m, par_m, x)
ind_prob

,indicador,modelo,empirico
0,P(X ≤ 30 kWh) – bajo consumo,0.6572,0.7020
1,P(X ≤ 100 kWh) – rango FOSE,0.9116,0.9271
2,P(X > 300 kWh) – alto consumo,0.0134,0.0287
3,Mediana (kWh),17.9209,17.9292
4,Percentil 90 (kWh),91.6002,74.0000
5,Percentil 99 (kWh),346.3673,839.3500
6,Media (kWh),40.2972,50.1027


## 4. Distribuciones por provincia
### 4.1 Comparación gráfica y prueba de Kruskal-Wallis
H₀: el consumo por cliente tiene la misma distribución en todas las provincias.

In [6]:
orden = b5.groupby("provincia")["kwh_cliente"].median().sort_values().index
fig = px.box(b5, x="provincia", y="kwh_cliente", category_orders={"provincia": list(orden)}, log_y=True,
             points=False, title="Consumo por cliente BT5B por provincia (escala log)")
fig.show()
kw = stats.kruskal(*[g["kwh_cliente"].to_numpy() for _, g in b5.groupby("provincia")])
print(f"Kruskal-Wallis H = {kw.statistic:,.1f}, p-valor = {kw.pvalue:.2e} → "
      + ("se rechaza H0: hay diferencias entre provincias." if kw.pvalue < 0.05 else "no se rechaza H0."))

Kruskal-Wallis H = 1,239.5, p-valor = 5.46e-258 → se rechaza H0: hay diferencias entre provincias.


### 4.2 Ajuste log-normal por provincia e intervalos de confianza bootstrap
Para cada provincia se estima una log-normal (μ, σ del logaritmo) y se calcula la probabilidad de estar en el rango FOSE.
El intervalo de confianza del **consumo medio ponderado por clientes** se obtiene con 1,000 réplicas *bootstrap*.

In [7]:
def boot_media_ponderada(v, w, B=1000):
    idx = rng.integers(0, len(v), size=(B, len(v)))
    medias = (v[idx] * w[idx]).sum(axis=1) / w[idx].sum(axis=1)
    return np.quantile(medias, [0.025, 0.975])

filas = []
for prov, g in b5.groupby("provincia"):
    v, w = g["kwh_cliente"].to_numpy(), g["clientes"].to_numpy()
    s, _, esc = stats.lognorm.fit(v, floc=0)
    lo, hi = boot_media_ponderada(v, w)
    filas.append({"provincia": prov, "n_registros": len(v), "clientes": w.sum(),
                  "mu_log": np.log(esc), "sigma_log": s, "mediana_modelo": esc,
                  "media_ponderada": (v * w).sum() / w.sum(), "IC95_inf": lo, "IC95_sup": hi,
                  "P_le_30": stats.lognorm.cdf(30, s, 0, esc), "P_le_100": stats.lognorm.cdf(100, s, 0, esc)})
prov_fit = pd.DataFrame(filas).sort_values("media_ponderada", ascending=False)
prov_fit

,provincia,n_registros,clientes,mu_log,sigma_log,mediana_modelo,media_ponderada,IC95_inf,IC95_sup,P_le_30,P_le_100
10,SAN ROMAN,953,4994937,3.4437,1.2433,31.3037,68.9803,67.3660,70.3334,0.4864,0.8249
9,SAN ANTONIO DE PUTINA,634,532551,3.1272,1.8256,22.8093,60.6361,52.1482,68.6026,0.5597,0.7909
8,PUNO,2047,3063034,2.9096,1.2338,18.3487,55.6952,52.1736,58.3082,0.6549,0.9153
6,MELGAR,1213,887995,3.0986,1.4365,22.1677,38.9634,35.3055,43.0382,0.5834,0.8529
1,CARABAYA,923,286190,3.2937,1.3884,26.9424,38.6987,36.3918,41.2013,0.5309,0.8276
11,SANDIA,830,275492,2.8448,1.2349,17.1975,36.7305,35.0529,38.1884,0.6739,0.9230
12,YUNGUYO,475,258802,3.6941,0.9572,40.2105,36.5300,35.7828,37.1900,0.3798,0.8294
2,CHUCUITO,686,763545,2.4826,0.9482,11.9727,25.8841,23.9465,27.6142,0.8337,0.9874
3,EL COLLAO,435,899194,2.8521,1.3784,17.3234,23.5157,22.9029,24.0250,0.6548,0.8983
5,LAMPA,1158,431980,2.7725,0.8574,15.9991,21.9167,21.1655,22.5970,0.7683,0.9837


In [8]:
fig = px.scatter(prov_fit, x="media_ponderada", y="provincia", error_x=prov_fit["IC95_sup"] - prov_fit["media_ponderada"],
                 error_x_minus=prov_fit["media_ponderada"] - prov_fit["IC95_inf"], size="clientes",
                 title="Consumo medio por cliente BT5B con IC 95% (bootstrap)",
                 labels={"media_ponderada": "kWh por cliente al mes"})
fig.update_yaxes(categoryorder="total ascending")
fig.show()

## 5. Detección de anomalías a partir de las distribuciones
- **Consumo atípico:** registros BT5B cuyo consumo por cliente cae fuera del intervalo [P0.5 ; P99.5] del modelo ajustado.
- **Precio atípico:** se calcula un *z-score robusto* (mediana y MAD) de `log(precio_kwh)` dentro de cada tarifa **de baja tensión**; |z| > 3.5 es anómalo.
  (En media tensión el CEA no recoge toda la energía facturada, así que su S/ por kWh no es comparable.)
- **Consumo sin facturación:** consumo > 0 pero monto facturado = 0.

Estas alertas se muestran en el dashboard para revisión (posibles errores de lectura, pérdidas o clientes especiales).

In [9]:
q_lo, q_hi = dist_m.ppf([0.005, 0.995], *par_m)
df["z_precio"] = np.nan
for tarifa, g in df[(df["precio_kwh"] > 0) & (df["nivel_tension"] == "BT")].groupby("tarifa"):
    lp = np.log(g["precio_kwh"])
    mad = stats.median_abs_deviation(lp, scale="normal")
    if mad > 0:
        df.loc[g.index, "z_precio"] = (lp - lp.median()) / mad

es_b5 = (df["tarifa"] == "BT5B") & (df["consumo_kwh"] > 0)
df["anom_consumo"] = es_b5 & ((df["kwh_cliente"] < q_lo) | (df["kwh_cliente"] > q_hi))
df["anom_precio"] = df["z_precio"].abs() > 3.5
df["anom_sin_factura"] = (df["consumo_kwh"] > 0) & (df["monto_soles"] == 0)
df["anomalia"] = df[["anom_consumo", "anom_precio", "anom_sin_factura"]].any(axis=1)

def tipo(r):
    t = []
    if r.anom_consumo: t.append("Consumo atípico")
    if r.anom_precio: t.append("Precio atípico")
    if r.anom_sin_factura: t.append("Consumo sin facturación")
    return ", ".join(t)
df["tipo_anomalia"] = df.apply(tipo, axis=1)
print(f"Umbrales de consumo atípico (BT5B): < {q_lo:.1f} o > {q_hi:.1f} kWh/cliente")
df.loc[df["anomalia"], "tipo_anomalia"].str.split(", ").explode().value_counts()

Umbrales de consumo atípico (BT5B): < 0.7 o > 475.8 kWh/cliente


tipo_anomalia
Precio atípico             1100
Consumo atípico             289
Consumo sin facturación     106
Name: count, dtype: int64

In [10]:
fig = px.scatter(df[df["precio_kwh"].between(0.01, 20)], x="kwh_cliente", y="precio_kwh", color="anomalia",
                 log_x=True, log_y=True, hover_data=["distrito", "tarifa", "clientes"], opacity=0.6,
                 color_discrete_map={True: "crimson", False: "lightgrey"},
                 title="Registros anómalos: consumo por cliente vs precio medio")
fig.show()

## 6. Distribución del tamaño de los distritos (ley rango-tamaño)
Si el número de clientes por distrito sigue una ley de potencia, en escala log-log la relación
rango-tamaño es aproximadamente lineal: log(clientes) = a − b·log(rango).

In [11]:
rs = ind[["distrito", "clientes"]].sort_values("clientes", ascending=False).reset_index(drop=True)
rs["rango"] = np.arange(1, len(rs) + 1)
reg = stats.linregress(np.log(rs["rango"]), np.log(rs["clientes"]))
fig = px.scatter(rs, x="rango", y="clientes", log_x=True, log_y=True, hover_name="distrito",
                 title=f"Rango-tamaño de distritos por clientes (pendiente b = {-reg.slope:.2f}, R² = {reg.rvalue**2:.2f})")
fig.add_scatter(x=rs["rango"], y=np.exp(reg.intercept + reg.slope * np.log(rs["rango"])), mode="lines",
                name="ajuste potencia")
fig.show()

## 7. Conclusiones y guardado

In [12]:
p100 = ind_prob.loc[1, "modelo"]
print(f"• El consumo por cliente residencial se ajusta mejor a una distribución {mejor} (menor AIC).")
print(f"• Según el modelo, el {p100:.0%} de los grupos de clientes BT5B consume ≤ 100 kWh/mes (rango FOSE) "
      f"y el {ind_prob.loc[0, 'modelo']:.0%} consume ≤ 30 kWh/mes.")
print(f"• Las provincias difieren significativamente (Kruskal-Wallis p = {kw.pvalue:.1e}); "
      f"{prov_fit.iloc[0]['provincia']} tiene el mayor consumo medio por cliente.")
print(f"• Se marcaron {df['anomalia'].sum():,} registros anómalos ({df['anomalia'].mean():.1%}) para revisión.")

resumen = {"variable": "kwh_cliente BT5B", "mejor_distribucion": mejor,
           "parametros": list(map(float, par_m)), "scipy_dist": dist_m.name,
           "umbral_anomalia": [float(q_lo), float(q_hi)],
           "indicadores": ind_prob.set_index("indicador")["modelo"].astype(float).to_dict(),
           "kruskal_p": float(kw.pvalue)}
json.dump(resumen, open(OUT / "distribuciones_resumen.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
tabla.to_csv(OUT / "comparacion_distribuciones.csv", index=False, encoding="utf-8-sig")
prov_fit.to_csv(OUT / "ajuste_provincia.csv", index=False, encoding="utf-8-sig")
df.to_parquet(OUT / "consumo_final.parquet", index=False)
df[df["anomalia"]].to_csv(OUT / "registros_anomalos.csv", index=False, encoding="utf-8-sig")
print("Guardado: distribuciones_resumen.json, comparacion_distribuciones.csv, ajuste_provincia.csv,",
      "consumo_final.parquet, registros_anomalos.csv")

• El consumo por cliente residencial se ajusta mejor a una distribución Log-normal (menor AIC).
• Según el modelo, el 91% de los grupos de clientes BT5B consume ≤ 100 kWh/mes (rango FOSE) y el 66% consume ≤ 30 kWh/mes.
• Las provincias difieren significativamente (Kruskal-Wallis p = 5.5e-258); SAN ROMAN tiene el mayor consumo medio por cliente.
• Se marcaron 1,418 registros anómalos (5.1%) para revisión.
Guardado: distribuciones_resumen.json, comparacion_distribuciones.csv, ajuste_provincia.csv, consumo_final.parquet, registros_anomalos.csv


## 8. Tablero final de KPIs para la Gerencia
Con las alertas de este notebook ya se pueden calcular los **12 KPIs** completos (ver definición en el Notebook 2, sección 11).
Esta es la vista resumida que se presenta al gerente; el dashboard interactivo (`python app.py`)
la replica en la pestaña **Resumen gerencial** y permite filtrarla por periodo, provincia y tarifa.

In [13]:
import importlib
import kpis
from plotly.subplots import make_subplots

importlib.reload(kpis)
tabla = kpis.tabla_kpis(df)
hay_previo = tabla["var_vs_mes_anterior_%"].notna().any()

fig = make_subplots(rows=4, cols=3, specs=[[{"type": "indicator"}] * 3] * 4, vertical_spacing=0.12)
for i, k in enumerate(kpis.CATALOGO):
    valor, var = tabla.loc[i, "valor"], tabla.loc[i, "var_vs_mes_anterior_%"]
    r, c = i // 3 + 1, i % 3 + 1
    sufijo = "%" if k["unidad"].startswith("%") else ""
    previo = valor / (1 + var / 100) if hay_previo and pd.notna(var) else None
    fig.add_trace(go.Indicator(
        mode="number+delta" if previo else "number", value=valor,
        number=dict(valueformat=k["fmt"], suffix=sufijo, font=dict(size=30)),
        delta=dict(reference=previo, relative=True, valueformat=".1%",
                   increasing_color="#1a7f37" if k["sentido"] >= 0 else "#b42318",
                   decreasing_color="#b42318" if k["sentido"] >= 0 else "#1a7f37") if previo else None,
        title=dict(text=f"<span style='font-size:11px;color:#52514e'>{k['bloque'].upper()}</span><br>"
                        f"<b>{k['nombre']}</b><br><span style='font-size:11px;color:#52514e'>{k['unidad']}</span>",
                   font=dict(size=14))), row=r, col=c)
periodo = str(df["periodo_cod"].max())
fig.update_layout(height=760, title=f"Tablero de KPIs · Electro Puno · periodo {periodo[:4]}-{periodo[4:]}"
                  + ("" if hay_previo else " (variación disponible al cargar 2+ meses)"),
                  margin=dict(t=90, b=20))
fig.show()

### Lectura para la Gerencia

In [14]:
k = kpis.calcular_kpis(df[df["periodo_cod"] == df["periodo_cod"].max()])
print(f"1. Comercial: se facturan S/ {k['facturacion']:,.0f} al mes por {k['energia_mwh']:,.0f} MWh "
      f"(S/ {k['precio_bt']:.3f} por kWh en baja tensión).")
print(f"2. Clientes: {k['clientes']:,.0f} suministros; cada uno aporta en promedio S/ {k['ingreso_cliente']:.2f} "
      f"y un hogar consume {k['kwh_residencial']:.1f} kWh al mes.")
print(f"3. Riesgo: el {k['pct_mt']:.1f}% del ingreso viene de media tensión (minería/industria) y el "
      f"{k['pct_top10']:.1f}% se concentra en 10 distritos → alta dependencia de pocos clientes y zonas.")
print(f"4. Control: hay S/ {k['monto_alerta']:,.0f} en {k['n_alertas']:,} registros con alerta para revisión "
      f"y el {k['pct_sin_consumo']:.1f}% de suministros no registra consumo.")
print(f"5. Social: el {k['pct_fose']:.1f}% de los hogares está en rango FOSE y el {k['pct_bajo_consumo']:.1f}% "
      f"consume ≤ 30 kWh (posible pobreza energética).")
kpis.tabla_kpis(df).to_csv(OUT / "kpis_gerencia.csv", index=False, encoding="utf-8-sig")

1. Comercial: se facturan S/ 34,968,838 al mes por 24,773 MWh (S/ 1.100 por kWh en baja tensión).
2. Clientes: 366,818 suministros; cada uno aporta en promedio S/ 95.33 y un hogar consume 52.9 kWh al mes.
3. Riesgo: el 33.7% del ingreso viene de media tensión (minería/industria) y el 80.6% se concentra en 10 distritos → alta dependencia de pocos clientes y zonas.
4. Control: hay S/ 229,068 en 28 registros con alerta para revisión y el 0.2% de suministros no registra consumo.
5. Social: el 98.2% de los hogares está en rango FOSE y el 34.3% consume ≤ 30 kWh (posible pobreza energética).


### Ver el dashboard dentro de Jupyter (opcional)
Para mostrar el dashboard interactivo al final de la presentación sin salir del notebook, descomenta y ejecuta:

In [15]:
from app import app
app.run(jupyter_mode="inline", port=8050)   # o jupyter_mode="tab" para abrirlo en otra pestaña